<a href="https://colab.research.google.com/github/ajzal4you/Master-Project-/blob/main/Classification_Breast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Breast Classification
ROOT = "/content/drive/MyDrive/MSc_Project/breast_classification"
import os
DATA_DIR   = os.path.join(ROOT, "data")
MODELS_DIR = os.path.join(ROOT, "models")
LOGS_DIR   = os.path.join(ROOT, "logs")
XAI_DIR = os.path.join(ROOT, "xai")
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("LOGS_DIR:", LOGS_DIR)


In [ ]:
!pip install -q kaggle
!pip install -q torch torchvision matplotlib
from google.colab import files
files.upload()  # Upload your kaggle.json here

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.9 MB/s eta 0:00:00


In [ ]:
!mkdir -p breast_mri_dataset/train/benign
!mkdir -p breast_mri_dataset/train/malignant
!mkdir -p breast_mri_dataset/validation/benign
!mkdir -p breast_mri_dataset/validation/malignant
!mkdir -p breast_mri_sorted
!mkdir -p ~/.kaggle

In [ ]:
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d uzairkhan45/breast-cancer-patients-mris
!unzip -o breast-cancer-patients-mris.zip -d breast_mri_dataset

In [ ]:
data_path = "/content/Breast Cancer Patients MRI's"

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import collections
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix,classification_report, ConfusionMatrixDisplay

In [ ]:
for root, dirs, files in os.walk(data_path):
    for d in dirs:
        print("📁", os.path.join(root, d))

In [ ]:
#Explore and Organize Folder
for root, dirs, files in os.walk("/content/breast_mri_dataset"):
    print("📁", root)
    for d in dirs:
        print("   └── 🗂️", d)
    for f in files[:3]:  # Only preview a few files
        print("   └── 📄", f)
    print("-" * 40)


In [ ]:
for root, dirs, files in os.walk("/content"):
    for d in dirs:
        print(os.path.join(root, d))

In [ ]:

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]) # Added normalization based on common practice
])

# Corrected base_dir to point to the 'train' directory which contains the class subdirectories in the unzipped dataset
base_dir = os.path.join('breast_mri_dataset', "Breast Cancer Patients MRI's", 'train')

dataset = datasets.ImageFolder(base_dir, transform=transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Total images:", len(dataset))
print("Class distribution:", dataset.class_to_idx)

In [ ]:
class BreastMRIClassifier(nn.Module):
    def __init__(self):
        super(BreastMRIClassifier, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
# Adjust the source path based on your dataset structure
!cp -r breast_mri_dataset/benign breast_mri_sorted/
!cp -r breast_mri_dataset/malignant breast_mri_sorted/

# Verify
!ls breast_mri_sorted
!ls breast_mri_sorted/benign | head

In [ ]:


# Define the transform (copied from DMPR461K3Jgk)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Define the dataset and dataloader (copied from DMPR461K3Jgk)
base_dir = os.path.join('breast_mri_dataset', "Breast Cancer Patients MRI's", 'train')
dataset = datasets.ImageFolder(base_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BreastMRIClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    running_loss = 0.0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"✅ Epoch {epoch+1}, Loss: {running_loss:.4f}")

In [ ]:
# Suppose you recorded the loss per epoch like this:
epochs = [1, 2, 3, 4, 5]
losses = [51.8445, 12.8347, 4.9401, 2.4530, 0.8706]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(epochs, losses, marker='o', linestyle='-', color='blue')
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.xticks(epochs)
plt.show()

In [ ]:
# Save the model state dictionary
torch.save(model.state_dict(), "breast_mri_model.pth")
print("✅ Model saved.")

# Load and evaluate model
model.load_state_dict(torch.load("breast_mri_model.pth"))
model.eval()

# If you want to print the structure
print(model)

In [ ]:
model = BreastMRIClassifier()
print(model)
print("________________________________________________________________")
print()
model.load_state_dict(torch.load("breast_mri_model.pth"))
model.eval()

In [ ]:

# Assuming 'device' is defined from previous cells (e.g., torch.device("cuda" if torch.cuda.is_available() else "cpu"))
# And 'dataloader' is defined from previous cells

# Get a batch
images, labels = next(iter(dataloader))
images, labels = images.to(device), labels.to(device)

# Ensure the model is on the correct device
model.to(device)

# Forward pass
with torch.no_grad():
    outputs = model(images)
    probs = torch.sigmoid(outputs)
    preds = (probs > 0.5).int().squeeze()

# Visualize 5 predictions
for i in range(5):
    # Denormalize and move back to CPU for plotting
    img_to_show = images[i].cpu()
    # Assuming normalization was with mean=[0.5] and std=[0.5]
    mean = torch.tensor([0.5]).view(-1, 1, 1)
    std = torch.tensor([0.5]).view(-1, 1, 1)
    img_to_show.mul_(std).add_(mean)
    img_to_show = torch.clamp(img_to_show, 0, 1)


    plt.imshow(img_to_show.permute(1, 2, 0))
    plt.title(f"True: {labels[i].item()}, Pred: {preds[i].item()}")
    plt.axis("off")
    plt.show()

In [ ]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).int().squeeze()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"✅ Accuracy: {accuracy:.2f}%")

In [ ]:
!ls breast_mri_dataset

In [ ]:
print("Total images:", len(dataset))
print("Class distribution:", dataset.class_to_idx)

In [ ]:
torch.save(model.state_dict(), "breast_mri_model.pth")
print("✅ Model saved.")

In [ ]:
# Denormalize the image
# Assuming the normalization was with mean=[0.5] and std=[0.5]
mean = torch.tensor([0.5]).view(-1, 1, 1) # Reshape for broadcasting
std = torch.tensor([0.5]).view(-1, 1, 1)  # Reshape for broadcasting
denormalized_img = images[0].clone().cpu()

# Apply denormalization using broadcasting
denormalized_img.mul_(std).add_(mean)

# Clip the pixel values to the valid range [0, 1] for display
denormalized_img = torch.clamp(denormalized_img, 0, 1)

plt.imshow(denormalized_img.permute(1, 2, 0))  # Convert to HWC
plt.title(f"Label: {labels[0]}")
plt.axis('off')
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    output = model(images.to(device))
    print("Output shape:", output.shape)
    print("Sample output:", output[0])

In [ ]:

os.makedirs("breast_mri_dataset/benign", exist_ok=True)
os.makedirs("breast_mri_dataset/malignant", exist_ok=True)

# Example move (adapt path to your case)
!mv "/content/Breast Cancer Patients MRI's/benign"/* "breast_mri_dataset/benign/"
!mv "/content/Breast Cancer Patients MRI's/malignant"/* "breast_mri_dataset/malignant/"


In [ ]:
def predict_image(path, model, class_names):
    model.eval()
    image = Image.open(path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)
    return class_names[predicted.item()]

# Example
# predict_image("breast_mri_sorted/benign/sample.jpg", model, dataset.classes)


In [ ]:
print("Lebels in dataset:",set([label for _,label in dataset]))
criterion = nn.CrossEntropyLoss()
print(collections.Counter([label for _, label in dataset]))

In [ ]:
img, label = dataset[0]
plt.imshow(img.permute(1, 2, 0))  # Unnormalize if needed
plt.title(f"Label: {label}")

In [ ]:
print("Model output sample:", outputs[:5].detach().cpu().numpy())
print(f"Sample Loss: {loss.item()}")

In [ ]:
  import torch
  import numpy as np
  from sklearn.metrics import confusion_matrix,classification_report
  from torchvision import datasets, transforms
  from torch.utils.data import DataLoader
  import os

  y_true = []
  y_pred = []

  # Define the transform (reusing the one used for training)
  transform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize([0.5], [0.5])
  ])

  # Define the path to the validation dataset
  validation_dir = os.path.join('breast_mri_dataset', "Breast Cancer Patients MRI's", 'validation')

  # Create the validation dataset
  validation_dataset = datasets.ImageFolder(validation_dir, transform=transform)

  # Create the test_loader (using the validation dataset)
  test_loader = DataLoader(validation_dataset, batch_size=16, shuffle=False) # Shuffle is typically False for evaluation


  model.eval()
  with torch.no_grad():
    for images,labels in test_loader:
      images = images.to(device)
      labels = labels.to(device)


      outputs = model(images)
      preds = torch.sigmoid(outputs) > 0.5 #Binary classification thresshold
      y_true.extend(labels.cpu().numpy())
      y_pred.extend(preds.cpu().numpy())

  # The rest of the evaluation code (e.g., confusion matrix, classification report)
  # will be added in a subsequent step after the data loading issue is resolved.